# 02 — Data Cleaning: Raw normalizado y export

Objetivo de este cuaderno:
- Tomar los archivos crudos (Excel) y generar dos tablas **staging** normalizadas para cargar a la BD.
- No toma decisiones de negocio (eso va en otros cuadernos/procesos).

Outputs (CSV):
- `02_data_cleaning/outputs/leads_raw_normalizado.csv`
- `02_data_cleaning/outputs/horas_raw_normalizado.csv`

In [11]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [12]:
# Configuración de archivos (ajusta si cambiás rutas)
# Nota: al ejecutar un notebook, el cwd suele ser la carpeta del notebook (02_data_cleaning/)
LEADS_FILE = Path('..') / '01_data_ingestion_enrichment' / 'leads.xlsx'
HORAS_FILE = Path('..') / '01_data_ingestion_enrichment' / 'proyectos_empresa.xlsx'

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('LEADS_FILE:', LEADS_FILE.resolve())
print('HORAS_FILE:', HORAS_FILE.resolve())
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())

LEADS_FILE: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\leads.xlsx
HORAS_FILE: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\01_data_ingestion_enrichment\proyectos_empresa.xlsx
OUTPUT_DIR: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs


In [13]:
def leer_archivo(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'No existe el archivo: {path.resolve()}')
    suffix = path.suffix.lower()
    if suffix in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    raise ValueError(f'Formato no soportado: {suffix}')

df_leads = leer_archivo(LEADS_FILE)
df_horas = leer_archivo(HORAS_FILE)

print('Leads:', df_leads.shape)
print('Horas:', df_horas.shape)

Leads: (440, 15)
Horas: (412, 18)


C:\Users\asus\AppData\Roaming\Python\Python312\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [14]:
# --- Normalización (staging) ---
def _strip_accents_any(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

def normalize_text_for_matching(value) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and np.isnan(value):
        return ''
    s = str(value).strip()
    if not s:
        return ''
    s = _strip_accents_any(s).upper()
    s = re.sub(r'[^A-Z0-9 ]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def normalize_colname(name) -> str:
    s = '' if name is None else str(name)
    s = _strip_accents_any(s).lower().strip()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s or 'col'

def _dedupe_colnames(colnames: list[str]) -> list[str]:
    seen: dict[str, int] = {}
    out: list[str] = []
    for c in colnames:
        if c not in seen:
            seen[c] = 1
            out.append(c)
        else:
            seen[c] += 1
            out.append(f'{c}_{seen[c]}')
    return out

def normalize_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = _dedupe_colnames([normalize_colname(c) for c in out.columns])
    return out

def find_col_by_candidates(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_to_actual = {normalize_colname(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_to_actual:
            return norm_to_actual[key]
    return None

In [15]:
# LEADS: quedarse solo con los campos requeridos + columnas *_norm
leads_required = {
    'empresa': ['Company', 'EMPRESA', 'Empresa', 'Razón Social', 'Razon Social', 'Nombre Empresa'],
    'lead_source': ['Lead Source', 'Fuente', 'Fuente Lead', 'Origen', 'Origen Lead'],
    'industry': ['Industry', 'Industria', 'Sector'],
    'no_of_employees': ['No. of Employees', 'No of Employees', 'Employees', 'N empleados', 'Numero de Empleados', 'Número de Empleados'],
    'annual_revenue': ['Annual Revenue', 'Revenue', 'Ingresos', 'Facturacion', 'Facturación'],
    'interes': ['Interés', 'Interes'],
    'cargo': ['Cargo', 'Puesto', 'Job Title', 'Position', 'Rol', 'Role'],
    'pais': ['País', 'Pais', 'Country'],
}

leads_cols_actual = {std: find_col_by_candidates(df_leads, cands) for std, cands in leads_required.items()}
missing = [k for k, v in leads_cols_actual.items() if v is None]
if missing:
    print('[AVISO] En LEADS no se detectaron estas columnas (se crearán como NaN):', missing)

df_leads_raw = pd.DataFrame({
    std: (df_leads[src].copy() if src is not None else np.nan)
    for std, src in leads_cols_actual.items()
})

# columnas *_norm solo para texto (para joins posteriores)
for c in ['empresa', 'lead_source', 'industry', 'interes', 'cargo', 'pais']:
    df_leads_raw[f'{c}_norm'] = df_leads_raw[c].map(normalize_text_for_matching)

# numéricos si vienen como texto
for c in ['no_of_employees', 'annual_revenue']:
    df_leads_raw[c] = pd.to_numeric(df_leads_raw[c], errors='coerce')

print('df_leads_raw:', df_leads_raw.shape)
display(df_leads_raw.head(5))

df_leads_raw: (440, 14)


,empresa,lead_source,industry,no_of_employees,annual_revenue,interes,cargo,pais,empresa_norm,lead_source_norm,industry_norm,interes_norm,cargo_norm,pais_norm
0,GAMEPLANET S.A. DE C.V.,NaN,NaN,NaN,NaN,Adaptive,Gerente General,NaN,GAMEPLANET S A DE C V,,,ADAPTIVE,GERENTE GENERAL,
1,ALCIONE MX,NaN,NaN,NaN,NaN,Adaptive,Encargado de Finanzas en TI,NaN,ALCIONE MX,,,ADAPTIVE,ENCARGADO DE FINANZAS EN TI,
2,ASTRAZENECA,NaN,NaN,NaN,NaN,Adaptive,Country Controller,NaN,ASTRAZENECA,,,ADAPTIVE,COUNTRY CONTROLLER,
3,FARMACIAS DEL AHORRO,NaN,NaN,NaN,NaN,Adaptive,Director del Centro de Servicios Compartidos,NaN,FARMACIAS DEL AHORRO,,,ADAPTIVE,DIRECTOR DEL CENTRO DE SERVICIOS COMPARTIDOS,
4,SANOFI,NaN,NaN,NaN,NaN,Adaptive,FP&A Manager,NaN,SANOFI,,,ADAPTIVE,FP A MANAGER,


In [16]:
# HORAS: normalizar headers de TODAS las columnas + empresa normalizada para staging
df_horas_raw = normalize_dataframe_columns(df_horas)

horas_company_candidates = [
    'EMPRESA', 'Empresa', 'Empresas', 'Company', 'CLIENTE', 'Cliente',
    'Razon Social', 'Razón Social', 'Nombre Empresa'
 ]

col_empresa_horas = find_col_by_candidates(df_horas_raw, horas_company_candidates)
if col_empresa_horas is None:
    print('[AVISO] No se pudo detectar columna de empresa en HORAS tras normalizar headers.')
else:
    if col_empresa_horas != 'empresa':
        df_horas_raw = df_horas_raw.rename(columns={col_empresa_horas: 'empresa'})

# Para staging en BD: conservar original y normalizar a MAYÚSCULAS en `empresa`
if 'empresa' in df_horas_raw.columns:
    df_horas_raw['empresa_raw'] = df_horas_raw['empresa']
    df_horas_raw['empresa'] = df_horas_raw['empresa_raw'].map(normalize_text_for_matching)
    # compatibilidad/claridad: mantener también empresa_norm
    df_horas_raw['empresa_norm'] = df_horas_raw['empresa']

# limpiar headers por si vinieran con espacios raros
df_horas_raw.columns = [str(c).strip() for c in df_horas_raw.columns]

print('df_horas_raw:', df_horas_raw.shape)
display(df_horas_raw.head(5))

df_horas_raw: (412, 20)


,empresa,id_pro,nombre,mostrar_listas,id_emp,lugar_por_defecto,id_esp,horas_estimadas,en_ejecucion,facturacion,ocupacion,horas_ejecutadas,fecha_corte,horas_ejecutadas_facturables,tipo_alcance,id_col_responsable,avance_real,avance_estimado,empresa_raw,empresa_norm
0,3DPHARMA,4298,GPF - Proyecto Necesidades AWS,1,3472,2,1,640.0,NaN,22500.0,Esporádico,NaN,NaT,NaN,Por Alcance,NaN,NaN,NaN,3dpharma,3DPHARMA
1,AVIS,364,Presupuesto Financiero,0,552,1,2,NaN,0.0,NaN,Stand by,672.00,2026-03-02 09:50:07,654.00,Por Alcance,NaN,NaN,NaN,AVIS,AVIS
2,AVIS,938,Presupuesto 2016,0,552,2,2,NaN,0.0,NaN,Stand by,368.00,2026-03-02 09:50:07,294.50,Por Alcance,NaN,NaN,NaN,AVIS,AVIS
3,ADIUM,2458,Soporte Filiales,0,4193,2,2,1440.0,0.0,NaN,Permanente,5256.48,2026-03-02 09:50:07,4945.33,Por Alcance,NaN,NaN,NaN,Adium,ADIUM
4,ADIUM,2397,Rollout Mexico,0,4193,2,2,415.0,0.0,NaN,Stand by,1159.00,2026-03-02 09:50:07,984.00,Por Alcance,NaN,NaN,NaN,Adium,ADIUM


In [17]:
# Export a CSV (staging para BD)
leads_out_path = OUTPUT_DIR / 'leads_raw_normalizado.csv'
horas_out_path = OUTPUT_DIR / 'horas_raw_normalizado.csv'

# Asegurar headers sin espacios
df_leads_raw.columns = [str(c).strip() for c in df_leads_raw.columns]
df_horas_raw.columns = [str(c).strip() for c in df_horas_raw.columns]

# utf-8-sig ayuda a que Excel/Windows lea bien acentos si abrís el CSV con doble click
df_leads_raw.to_csv(leads_out_path, index=False, encoding='utf-8-sig')
df_horas_raw.to_csv(horas_out_path, index=False, encoding='utf-8-sig')

print('Export OK')
print('- LEADS:', leads_out_path.resolve())
print('- HORAS:', horas_out_path.resolve())

Export OK
- LEADS: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\leads_raw_normalizado.csv
- HORAS: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\horas_raw_normalizado.csv


In [18]:
# --- Export: union all de EMPRESAS (una sola columna) ---

def _prep_distinct_series(df: pd.DataFrame, col: str | None) -> pd.Series:
    if not col or col not in df.columns:
        return pd.Series(dtype='string')
    s = df[col].dropna().astype('string')
    s = s.str.strip()
    s = s[s != '']
    # distinct dentro de cada fuente (tal como pediste)
    return s.drop_duplicates()

# LEADS: el campo "Company" ya fue estandarizado a `empresa` y normalizado a `empresa_norm`
leads_col = 'empresa_norm' if 'empresa_norm' in df_leads_raw.columns else ('empresa' if 'empresa' in df_leads_raw.columns else None)
# HORAS: `empresa_norm` existe si se detectó empresa y se normalizó
horas_col = 'empresa_norm' if 'empresa_norm' in df_horas_raw.columns else ('empresa' if 'empresa' in df_horas_raw.columns else None)

leads_empresas = _prep_distinct_series(df_leads_raw, leads_col)
horas_empresas = _prep_distinct_series(df_horas_raw, horas_col)

# union all (concat) luego de distinct por separado
empresas_union = pd.concat([horas_empresas, leads_empresas], ignore_index=True)

# Para obtener "un solo conjunto" final, deduplicar globalmente
empresas_union = empresas_union.drop_duplicates().reset_index(drop=True)

df_empresas_union_all = pd.DataFrame({'empresa_norm': empresas_union})

# Export CSV (una sola columna)
empresas_out = OUTPUT_DIR / 'empresas_union_all.csv'
df_empresas_union_all.to_csv(empresas_out, index=False, encoding='utf-8-sig')

print('df_empresas_union_all:', df_empresas_union_all.shape)
display(df_empresas_union_all.head(30))
print('Export OK:', empresas_out.resolve())

df_empresas_union_all: (447, 1)


,empresa_norm
0,3DPHARMA
1,AVIS
2,ADIUM
3,ALMEXA
4,ALPER SEGUROS
5,ARAUCO
6,ASEGURADORA DEL SUR
7,ASESORIA Y CONTROL
8,AUTOSHARE
9,BAC


Export OK: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\empresas_union_all.csv
